In [1]:
"""Lab6 tutorial: PyTorch basics (tensors, autograd, nn modules)

File thực hiện các task yêu cầu:
- Phần 1: Khám phá Tensor (tạo, phép toán, indexing, reshape)
- Phần 2: Autograd (tính đạo hàm, hiệu ứng gọi backward nhiều lần)
- Phần 3: Torch NN (Linear, Embedding, định nghĩa nn.Module)

"""


import torch
import numpy as np


def part1_tensors():
    print("=== Phần 1: Khám phá Tensor ===")

    # Task 1.1: Tạo Tensor
    data = [[1, 2], [3, 4]]
    x_data = torch.tensor(data)
    print(f"Tensor từ list:\n {x_data}\n")

    np_array = np.array(data)
    x_np = torch.from_numpy(np_array)
    print(f"Tensor từ NumPy array:\n {x_np}\n")

    x_ones = torch.ones_like(x_data)
    print(f"Ones Tensor:\n {x_ones}\n")

    x_rand = torch.rand_like(x_data, dtype=torch.float)
    print(f"Random Tensor:\n {x_rand}\n")

    print(f"Shape của tensor: {x_rand.shape}")
    print(f"Datatype của tensor: {x_rand.dtype}")
    print(f"Device lưu trữ tensor: {x_rand.device}\n")

    # Task 1.2: Phép toán
    print("Task 1.2: Phép toán")
    print("x_data + x_data:\n", x_data + x_data)
    print("x_data * 5:\n", x_data * 5)
    # matrix multiplication
    print("x_data @ x_data.T:\n", x_data @ x_data.t())
    print()

    # Task 1.3: Indexing và slicing
    print("Task 1.3: Indexing và slicing")
    print("Hàng đầu tiên:\n", x_data[0])
    print("Cột thứ hai:\n", x_data[:, 1])
    print("Giá trị tại hàng 2, cột 2:\n", x_data[1, 1])
    print()

    # Task 1.4: Thay đổi shape
    print("Task 1.4: Thay đổi shape")
    r = torch.rand(4, 4)
    print("Ban đầu shape:", r.shape)
    r_flat = r.view(16, 1)
    print("Sau view(16,1):", r_flat.shape)
    print()

part1_tensors()

=== Phần 1: Khám phá Tensor ===
Tensor từ list:
 tensor([[1, 2],
        [3, 4]])

Tensor từ NumPy array:
 tensor([[1, 2],
        [3, 4]])

Ones Tensor:
 tensor([[1, 1],
        [1, 1]])

Random Tensor:
 tensor([[0.9910, 0.8323],
        [0.8317, 0.1712]])

Shape của tensor: torch.Size([2, 2])
Datatype của tensor: torch.float32
Device lưu trữ tensor: cpu

Task 1.2: Phép toán
x_data + x_data:
 tensor([[2, 4],
        [6, 8]])
x_data * 5:
 tensor([[ 5, 10],
        [15, 20]])
x_data @ x_data.T:
 tensor([[ 5, 11],
        [11, 25]])

Task 1.3: Indexing và slicing
Hàng đầu tiên:
 tensor([1, 2])
Cột thứ hai:
 tensor([2, 4])
Giá trị tại hàng 2, cột 2:
 tensor(4)

Task 1.4: Thay đổi shape
Ban đầu shape: torch.Size([4, 4])
Sau view(16,1): torch.Size([16, 1])



In [7]:
def part2_autograd():
    print("=== Phần 2: Tự động tính đạo hàm với autograd ===")
    x = torch.ones(1, requires_grad=True)
    print(f"x: {x}")

    y = x + 2
    print(f"y = x + 2 -> {y}")
    print(f"grad_fn của y: {y.grad_fn}")

    z = y * y * 3
    print(f"z = 3*(x+2)^2 -> {z}")
    # ----- Ví dụ 1: gọi backward 2 lần mà KHÔNG dùng retain_graph (sẽ gây RuntimeError) -----
    print('\nVí dụ A: backward hai lần KHÔNG dùng retain_graph (sẽ gây lỗi)')
    x1 = torch.ones(1, requires_grad=True)
    y1 = x1 + 2
    z1 = 3 * (y1 ** 2)
    z1.backward()
    print("Lần 1: x1.grad =", x1.grad)
    try:
        # Lần 2: đồ thị đã bị giải phóng, nên sẽ ném RuntimeError
        z1.backward()
    except RuntimeError as e:
        print("Lần 2 gây RuntimeError như sau (ví dụ lỗi bạn thấy):")
        print(e)

    # ----- Ví dụ 2: dùng retain_graph=True để cho phép backward nhiều lần -----
    print('\nVí dụ B: dùng retain_graph=True để backward nhiều lần (grad sẽ cộng dồn)')
    x2 = torch.ones(1, requires_grad=True)
    y2 = x2 + 2
    z2 = 3 * (y2 ** 2)
    z2.backward(retain_graph=True)  # giữ graph để có thể backward lần nữa
    print("Sau lần 1 (retain_graph=True): x2.grad =", x2.grad)
    # Gọi lại backward mà không zero -> gradient cộng dồn
    z2.backward(retain_graph=True)
    print("Sau lần 2 (cộng dồn): x2.grad =", x2.grad)
    # Reset grad về 0
    x2.grad.zero_()
    print("Sau x2.grad.zero_():", x2.grad)

    # ----- Ví dụ 3: tính lại forward (recompute) và backward lại — cách an toàn về bộ nhớ -----
    print('\nVí dụ C: recompute forward rồi backward (khuyến nghị khi không cần giữ graph)')
    x3 = torch.ones(1, requires_grad=True)
    y3 = x3 + 2
    z3 = 3 * (y3 ** 2)
    z3.backward()
    print('Lần 1: x3.grad =', x3.grad)
    x3.grad.zero_()
    # tính lại z3 (tạo đồ thị mới)
    y3 = x3 + 2
    z3 = 3 * (y3 ** 2)
    z3.backward()
    print('Sau recompute và backward: x3.grad =', x3.grad)

    # ----- Ví dụ 4: đạo hàm bậc hai (create_graph=True) -----
    print('\nVí dụ D: đạo hàm bậc hai bằng create_graph=True')
    x4 = torch.ones(1, requires_grad=True)
    y4 = x4 + 2
    z4 = 3 * (y4 ** 2)
    dz_dx = torch.autograd.grad(z4, x4, create_graph=True)[0]
    # dz_dx = 6*(x+2); đạo hàm bậc hai là 6
    d2 = torch.autograd.grad(dz_dx, x4)[0]
    print('dz/dx =', dz_dx)
    print('d2z/dx2 =', d2)

part2_autograd()

=== Phần 2: Tự động tính đạo hàm với autograd ===
x: tensor([1.], requires_grad=True)
y = x + 2 -> tensor([3.], grad_fn=<AddBackward0>)
grad_fn của y: <AddBackward0 object at 0x7ee729ae17e0>
z = 3*(x+2)^2 -> tensor([27.], grad_fn=<MulBackward0>)

Ví dụ A: backward hai lần KHÔNG dùng retain_graph (sẽ gây lỗi)
Lần 1: x1.grad = tensor([18.])
Lần 2 gây RuntimeError như sau (ví dụ lỗi bạn thấy):
Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

Ví dụ B: dùng retain_graph=True để backward nhiều lần (grad sẽ cộng dồn)
Sau lần 1 (retain_graph=True): x2.grad = tensor([18.])
Sau lần 2 (cộng dồn): x2.grad = tensor([36.])
Sau x2.grad.zero_(): tensor([0.])

Ví dụ C: recompute forward rồi bac

In [4]:
def part3_nn():
    print("=== Phần 3: Xây dựng mô hình với torch.nn ===")

    # Task 3.1: nn.Linear
    linear_layer = torch.nn.Linear(in_features=5, out_features=2)
    input_tensor = torch.randn(3, 5)
    output = linear_layer(input_tensor)
    print(f"Input shape: {input_tensor.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Output:\n{output}\n")

    # Task 3.2: nn.Embedding
    embedding_layer = torch.nn.Embedding(num_embeddings=10, embedding_dim=3)
    input_indices = torch.LongTensor([1, 5, 0, 8])
    embeddings = embedding_layer(input_indices)
    print(f"Input indices: {input_indices}")
    print(f"Embeddings shape: {embeddings.shape}")
    print(f"Embeddings:\n{embeddings}\n")

    # Task 3.3: Kết hợp thành một nn.Module
    from torch import nn

    class MyFirstModel(nn.Module):
        def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
            super(MyFirstModel, self).__init__()
            self.embedding = nn.Embedding(vocab_size, embedding_dim)
            # Áp dụng linear token-wise (như ví dụ đơn giản)
            self.linear = nn.Linear(embedding_dim, hidden_dim)
            self.activation = nn.ReLU()
            self.output_layer = nn.Linear(hidden_dim, output_dim)

        def forward(self, indices):
            # indices: (batch, seq_len)
            embeds = self.embedding(indices)  # (batch, seq_len, embedding_dim)
            # Áp linear cho từng token
            hidden = self.activation(self.linear(embeds))  # (batch, seq_len, hidden_dim)
            output = self.output_layer(hidden)  # (batch, seq_len, output_dim)
            return output

    model = MyFirstModel(vocab_size=100, embedding_dim=16, hidden_dim=8, output_dim=2)
    input_data = torch.LongTensor([[1, 2, 5, 9]])  # (1, 4)
    output_data = model(input_data)
    print(f"Model output shape: {output_data.shape}")
    print(f"Model output:\n{output_data}\n")

part3_nn()

=== Phần 3: Xây dựng mô hình với torch.nn ===
Input shape: torch.Size([3, 5])
Output shape: torch.Size([3, 2])
Output:
tensor([[ 0.5352, -0.4765],
        [-0.0046, -0.7554],
        [ 1.2330, -0.1576]], grad_fn=<AddmmBackward0>)

Input indices: tensor([1, 5, 0, 8])
Embeddings shape: torch.Size([4, 3])
Embeddings:
tensor([[-0.0426, -0.9307,  2.4297],
        [ 0.1318,  0.2912,  0.5955],
        [-1.9064, -0.3147,  1.6491],
        [-1.9698,  0.4928,  0.4350]], grad_fn=<EmbeddingBackward0>)

Model output shape: torch.Size([1, 4, 2])
Model output:
tensor([[[-0.0972,  0.0823],
         [-0.1021,  0.2458],
         [-0.0492,  0.0206],
         [ 0.1038,  0.4007]]], grad_fn=<ViewBackward0>)

